In [ ]:
import os

from dotenv import load_dotenv

from comment import CommentRepository
from util import to_text_without_leading_common_whitespace

load_dotenv()
import jpype.imports
from jpype import JClass
from db_config import SessionLocal
from repository import Repository
session = SessionLocal()
commentDao = CommentRepository()
repositoryDao = Repository()
def get_method_extractor():
    if not jpype.isJVMStarted():
        jar_path = os.getenv('COMMENT_SCANNER_JAR')
        dependency_path = os.getenv('SATD_DETECTOR_DEPENDENCY')
        jpype.startJVM(
            jpype.getDefaultJVMPath(),
            classpath=[jar_path, dependency_path]
        )

    from ca.sqlmlab.comment.scanner import MethodExtractor
    return MethodExtractor()
NUMBER_OF_LINES = 10
method_extractor = get_method_extractor()
while True:
    comments = commentDao.get_comments_having_null_code(limit=100)
    # comments = [commentDao.get_comment(775)]
    for comment in comments:
        repositoryEntity = repositoryDao.get_repository(comment.repository_id)
        file_location = os.getenv(
            'REPOSITORY_DIRECTORY') + '/' + comment.repository_directory + '/' + comment.file

        url = f'{repositoryEntity.repo_url}/blob/{repositoryEntity.commit_hash}/{comment.file}/#L{comment.start_line}'
        print(f'\n\n\n\n########################## {comment.id} #########################')
        print(
            f'Repository: {comment.repository_directory}\nFile:\n{file_location}:{comment.start_line}\nURL: {url}\nComment:\n\n{comment.text}\n')

        start_line_index = comment.start_line - 1
        end_line_index = comment.end_line - 1
        with open(file_location, 'r') as file:
            lines = file.readlines()
            lines_before = lines[max(0, start_line_index - NUMBER_OF_LINES):start_line_index]
            code_before = to_text_without_leading_common_whitespace(lines_before)

            lines_after_begin_index = start_line_index if comment.text in lines[
                start_line_index] else end_line_index + 1
            lines_after = lines[lines_after_begin_index: min(lines_after_begin_index + NUMBER_OF_LINES, len(lines))]
            code_after = to_text_without_leading_common_whitespace(lines_after)

            method_code = str(method_extractor.extractMethod(file_location, comment.start_line))
            comment.code_method = method_code.replace("\x00", "")
            comment.code_before = code_before.replace("\x00", "")
            comment.code_after = code_after.replace("\x00", "")
        session.merge(comment)
        session.commit()
    if len(comments) == 0:
        break







########################## 1150963 #########################
Repository: apache--hive
File:
/home/cs/grad/islams32/dev/rnd/technical-debt/repository/apache--hive/service/src/test/org/apache/hive/service/auth/ldap/TestLdapSearch.java:333
URL: https://github.com/apache/hive/blob/b914b6a29ba0b1fab5e64034ee289e2deaa1a844/service/src/test/org/apache/hive/service/auth/ldap/TestLdapSearch.java/#L333
Comment:

//contains )





########################## 13 #########################
Repository: iluwatar--java-design-patterns
File:
/home/cs/grad/islams32/dev/rnd/technical-debt/repository/iluwatar--java-design-patterns/leader-followers/src/test/java/com/iluwatar/leaderfollowers/WorkCenterTest.java:32
URL: https://github.com/iluwatar/java-design-patterns/blob/9ad38a4f3ca3d26b1fe9eb9b13f10591a802adeb/leader-followers/src/test/java/com/iluwatar/leaderfollowers/WorkCenterTest.java/#L32
Comment:

/**
 * Tests for WorkCenter
 */





########################## 15 #########################
Reposito